# All of Us tutorial: extract triglycerides, HDL, and build a TG:HDL phenotype

**Goal:** run this notebook inside an **All of Us Researcher Workbench Jupyter environment** to:

1. Discover your active workspace bucket, CDR dataset, and notebook root directory.
2. Inspect the active OMOP CDR tables/schema.
3. Identify triglyceride and HDL cholesterol measurement concepts from the active CDR.
4. Extract TG and HDL lab values from the OMOP `measurement` table.
5. Convert units to mg/dL when possible.
6. Build same-day TG:HDL ratio phenotypes at the person level.
7. Save analysis-ready phenotype files locally and optionally copy them to your workspace bucket.

This tutorial is intentionally written to avoid hard-coding your workspace path or CDR version. It should work when copied across AoU workspaces, as long as the environment exposes a CDR dataset and you have access to BigQuery.

## Important privacy note

The output files contain person-level participant identifiers. Keep them inside the AoU controlled environment / workspace bucket. Do **not** download or export participant-level data outside approved AoU systems.

## Cell 1 — Robust AoU Workbench setup

Run this first. It is self-contained and does four things:

- discovers the active Google project from environment variables, `gcloud`, or BigQuery defaults;
- discovers a queryable AoU CDR, preferring Controlled Tier v9;
- creates an organized local project directory under `/home/jupyter/aou_tg_hdl_ratio` by default;
- discovers and verifies a writable GCS root, usually from buckets visible in your Google project.

Optional overrides, if automatic discovery ever fails:

```python
import os
os.environ["AOU_CDR_OVERRIDE"] = "wb-silky-artichoke-2408.C2025Q4R6"
os.environ["AOU_TGHDL_GCS_ROOT"] = "gs://dataproc-staging-wb-swift-parsnip-1270/aou_tg_hdl_ratio"
os.environ["AOU_TGHDL_REQUIRE_GCS"] = "0"  # set 1 only when you want to stop if GCS backup is unavailable
```
You can probably find yours in the workspace UI.

In [ ]:
# Cell 1: Robust All of Us Workbench 2.0 setup for TG:HDL phenotype extraction.
# This cell discovers a queryable CDRv9 dataset, creates organized local folders,
# discovers a writable GCS root, verifies backup writes, and defines save/upload helpers.

import os
import sys
import re
import json
import math
import time
import shutil
import subprocess
import uuid
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# -----------------------------
# User-tunable defaults.
# -----------------------------
PROJECT_SUBDIR = "aou_tg_hdl_ratio"

# Prefer Controlled Tier v9. Registered Tier v9 is a fallback for basic EHR-only extraction.
DEFAULT_CDR_CANDIDATES = [
    "wb-silky-artichoke-2408.C2025Q4R6",  # Controlled Tier CDRv9
    "wb-affable-acorn-7941.R2025Q4R6",    # Registered Tier CDRv9 fallback
]

PROJECT_ROOT = Path(os.environ.get("AOU_TGHDL_PROJECT_ROOT", str(Path.home() / PROJECT_SUBDIR)))
REQUIRE_GCS_BACKUP = os.environ.get("AOU_TGHDL_REQUIRE_GCS", "0").strip() not in {"0", "false", "False", "no", "NO"}
AUTO_UPLOAD_SAVED_FILES = os.environ.get("AOU_TGHDL_AUTO_UPLOAD", "1").strip() not in {"0", "false", "False", "no", "NO"}

# Optional overrides. These are safe to set in a small cell above this one if needed.
# Example:
#   os.environ["AOU_CDR_OVERRIDE"] = "wb-silky-artichoke-2408.C2025Q4R6"
#   os.environ["AOU_TGHDL_GCS_ROOT"] = "gs://your-writable-bucket/aou_tg_hdl_ratio"
USER_CDR_OVERRIDE = os.environ.get("AOU_CDR_OVERRIDE", "").strip()
USER_GCS_ROOT = os.environ.get("AOU_TGHDL_GCS_ROOT", "").strip()

# -----------------------------
# Basic shell/environment helpers.
# -----------------------------
def _run(cmd, check=False, quiet=True, timeout=120):
    """Run a shell command and return stdout, stderr, returncode."""
    p = subprocess.run(
        cmd,
        shell=True,
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=timeout,
    )
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}")
    if not quiet:
        print(p.stdout)
        if p.stderr:
            print(p.stderr)
    return p.stdout.strip(), p.stderr.strip(), p.returncode


def _load_exports_from_bashrc():
    """Recover Workbench variables that may only be exported in ~/.bashrc."""
    bashrc = Path.home() / ".bashrc"
    if not bashrc.exists():
        return
    for line in bashrc.read_text(errors="ignore").splitlines():
        line = line.strip()
        if not line.startswith("export ") or "=" not in line:
            continue
        name, value = line.replace("export ", "", 1).split("=", 1)
        name = name.strip()
        value = value.strip().strip('"').strip("'")
        if name and name not in os.environ and name != "PATH":
            os.environ[name] = value

_load_exports_from_bashrc()


def _discover_project():
    """Find the active Google Cloud project from env, gcloud, or BigQuery defaults."""
    for key in ["GOOGLE_CLOUD_PROJECT", "GOOGLE_PROJECT", "GCP_PROJECT", "PROJECT_ID"]:
        if os.environ.get(key):
            return os.environ[key]
    out, err, code = _run("gcloud config get-value project 2>/dev/null", timeout=30)
    if code == 0 and out and out != "(unset)":
        return out
    try:
        from google.cloud import bigquery
        return bigquery.Client().project
    except Exception:
        return None

PROJECT = _discover_project()
if PROJECT is None:
    raise RuntimeError("Could not determine the active Google Cloud project. Restart JupyterLab, then rerun this cell.")
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT

WORKSPACE_NAMESPACE = os.environ.get("WORKSPACE_NAMESPACE", "unknown_namespace")
WORKSPACE_NAME = os.environ.get("WORKSPACE_NAME", "unknown_workspace")
WORKSPACE_BUCKET_ENV = os.environ.get("WORKSPACE_BUCKET") or os.environ.get("GOOGLE_BUCKET")

# -----------------------------
# Local project directory layout.
# -----------------------------
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
DERIVED_DIR = DATA_DIR / "derived"
QC_DIR = PROJECT_ROOT / "qc"
RESULTS_DIR = PROJECT_ROOT / "results"
PLOTS_DIR = PROJECT_ROOT / "plots"
LOG_DIR = PROJECT_ROOT / "logs"
TMP_DIR = PROJECT_ROOT / "tmp"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RESULTS_DIR / RUN_ID

for d in [PROJECT_ROOT, DATA_DIR, RAW_DIR, INTERMEDIATE_DIR, DERIVED_DIR, QC_DIR, RESULTS_DIR, PLOTS_DIR, LOG_DIR, TMP_DIR, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------
# CDR discovery and verification.
# -----------------------------
def normalize_cdr_value(value):
    """Normalize CDR values to a BigQuery dataset path: project.dataset."""
    if value is None:
        return None
    value = str(value).strip().strip("`").strip('"').strip("'")
    if not value:
        return None
    m = re.search(r"projects/([^/]+)/datasets/([^/\s]+)", value)
    if m:
        return f"{m.group(1)}.{m.group(2)}"
    if re.fullmatch(r"C\d{4}Q\d+R\d+", value):
        return f"wb-silky-artichoke-2408.{value}"
    if re.fullmatch(r"R\d{4}Q\d+R\d+", value):
        return f"wb-affable-acorn-7941.{value}"
    if re.fullmatch(r"[A-Za-z0-9_-]+\.[A-Za-z0-9_]+", value):
        return value
    return None


def _candidate_cdrs():
    candidates = []
    if USER_CDR_OVERRIDE:
        candidates.append(("AOU_CDR_OVERRIDE", normalize_cdr_value(USER_CDR_OVERRIDE)))
    for key in [
        "WORKSPACE_CDR", "AOU_CDR", "CDR", "CDR_DATASET", "CDR_DATASET_ID", "CDR_VERSION",
        "AOU_CDR_VERSION", "WORKSPACE_DATASET", "DATA_COLLECTION", "AOU_DATA_COLLECTION",
    ]:
        cdr = normalize_cdr_value(os.environ.get(key))
        if cdr:
            candidates.append((key, cdr))
    for cdr in DEFAULT_CDR_CANDIDATES:
        candidates.append(("known_workbench2_cdrv9", cdr))
    seen = set()
    out = []
    for source, cdr in candidates:
        if cdr and cdr not in seen:
            seen.add(cdr)
            out.append((source, cdr))
    return out

try:
    from google.cloud import bigquery
    bq_client = bigquery.Client(project=PROJECT)
except Exception as e:
    raise RuntimeError(
        "Could not initialize google.cloud.bigquery.Client. Restart/authenticate the Workbench JupyterLab app. "
        f"Original error: {repr(e)}"
    )

cdr_diagnostics = []
CDR = None
cdr_source = None
for source, candidate in _candidate_cdrs():
    try:
        bq_client.query(f"SELECT 1 AS ok FROM `{candidate}.person` LIMIT 1").result(max_results=1)
        CDR = candidate
        cdr_source = source
        cdr_diagnostics.append({"source": source, "candidate": candidate, "queryable": True, "error": None})
        break
    except Exception as e:
        cdr_diagnostics.append({"source": source, "candidate": candidate, "queryable": False, "error": repr(e)[:1000]})

pd.DataFrame(cdr_diagnostics).to_csv(LOG_DIR / "cdr_autodiscovery_diagnostics.csv", index=False)
if CDR is None:
    display(pd.DataFrame(cdr_diagnostics))
    raise RuntimeError(
        "No queryable All of Us CDR was found. In Workbench 2.0, add an AoU CDR data resource to the workspace, "
        "or set os.environ['AOU_CDR_OVERRIDE']='project.dataset' above this cell."
    )
os.environ["AOU_CDR_OVERRIDE"] = CDR

# Alias used by later cells.
client = bq_client

# -----------------------------
# GCS workspace bucket/root discovery and write verification.
# -----------------------------
def _normalize_gs_uri(value, append_project_subdir=True):
    """Normalize a bucket or gs:// URI into a GCS root for this project."""
    if value is None:
        return None
    value = str(value).strip().strip('"').strip("'").rstrip("/")
    if not value:
        return None
    if not value.startswith("gs://"):
        value = "gs://" + value
    if append_project_subdir and not value.rstrip("/").endswith("/" + PROJECT_SUBDIR):
        value = value.rstrip("/") + "/" + PROJECT_SUBDIR
    return value


def _candidate_gcs_roots():
    roots = []
    # 1) Respect explicit user/root settings first.
    if USER_GCS_ROOT:
        roots.append(("AOU_TGHDL_GCS_ROOT", _normalize_gs_uri(USER_GCS_ROOT, append_project_subdir=False)))
    # 2) Common Workbench/Terra/env bucket variables.
    for key in [
        "WORKSPACE_BUCKET", "GOOGLE_BUCKET", "WORKSPACE_STORAGE_BUCKET", "TERRA_WORKSPACE_BUCKET",
        "WORKSPACE_GCS_BUCKET", "GCS_BUCKET", "BUCKET", "WORKSPACE_BUCKET_URL",
    ]:
        if os.environ.get(key):
            roots.append((key, _normalize_gs_uri(os.environ[key], append_project_subdir=True)))
    # 3) Other env vars containing bucket-looking values.
    for key, value in os.environ.items():
        if "BUCKET" in key.upper() and value and "gs://" in value:
            roots.append((key, _normalize_gs_uri(value, append_project_subdir=True)))
    # 4) Buckets visible in this GCP project. Prefer non-temp buckets, but test all.
    out, err, code = _run(f"gcloud storage buckets list --project={PROJECT} --format='value(name)'", timeout=60)
    if code == 0:
        bucket_names = [x.strip() for x in out.splitlines() if x.strip()]
        def score(name):
            n = name.lower()
            s = 0
            if PROJECT.lower() in n: s += 30
            if "workspace" in n or "wb-" in n or "aou" in n: s += 20
            if "staging" in n: s += 8
            if "temp" in n: s -= 30
            return -s, n
        for b in sorted(bucket_names, key=score):
            roots.append(("gcloud_bucket_list", _normalize_gs_uri(b, append_project_subdir=True)))
    # Deduplicate.
    seen = set()
    out_roots = []
    for source, root in roots:
        if root and root not in seen:
            seen.add(root)
            out_roots.append((source, root))
    return out_roots


def _copy_to_gcs(local_path, gcs_path, recursive=False, timeout=300):
    """Copy using gcloud storage, with gsutil fallback."""
    local_path = str(local_path)
    gcs_path = str(gcs_path)
    if shutil.which("gcloud"):
        flag = "--recursive" if recursive else ""
        out, err, code = _run(f"gcloud storage cp {flag} '{local_path}' '{gcs_path}'", timeout=timeout)
        if code == 0:
            return True, None
        last_err = err or out
    else:
        last_err = "gcloud not found"
    if shutil.which("gsutil"):
        flag = "-r" if recursive else ""
        out, err, code = _run(f"gsutil -m cp {flag} '{local_path}' '{gcs_path}'", timeout=timeout)
        if code == 0:
            return True, None
        last_err = err or out
    return False, str(last_err)[:1000]


def _remove_gcs(gcs_path, timeout=120):
    if shutil.which("gcloud"):
        _run(f"gcloud storage rm '{gcs_path}'", timeout=timeout)
    elif shutil.which("gsutil"):
        _run(f"gsutil rm '{gcs_path}'", timeout=timeout)


def _test_gcs_root(root):
    """Verify that a candidate GCS root is writable by copying and removing a tiny file."""
    test_id = uuid.uuid4().hex[:12]
    local_test = Path("/tmp") / f"aou_tghdl_write_test_{test_id}.txt"
    local_test.write_text(f"aou_tg_hdl_ratio write test {datetime.now(timezone.utc).isoformat()}\n")
    remote_test = root.rstrip("/") + f"/logs/_write_tests/{local_test.name}"
    ok, err = _copy_to_gcs(local_test, remote_test, recursive=False, timeout=120)
    if ok:
        _remove_gcs(remote_test, timeout=120)
        return True, None
    return False, err

GCS_ROOT = None
gcs_source = None
gcs_diagnostics = []
for source, root in _candidate_gcs_roots():
    ok, err = _test_gcs_root(root)
    gcs_diagnostics.append({"source": source, "candidate_gcs_root": root, "writable": ok, "error": err})
    if ok:
        GCS_ROOT = root.rstrip("/")
        gcs_source = source
        break

pd.DataFrame(gcs_diagnostics).to_csv(LOG_DIR / "gcs_autodiscovery_diagnostics.csv", index=False)
if GCS_ROOT is None and REQUIRE_GCS_BACKUP:
    display(pd.DataFrame(gcs_diagnostics))
    raise RuntimeError(
        "No writable GCS root was found. Because AOU_TGHDL_REQUIRE_GCS=1, stopping. "
        "Set os.environ['AOU_TGHDL_GCS_ROOT']='gs://bucket/aou_tg_hdl_ratio' above this cell, "
        "or set AOU_TGHDL_REQUIRE_GCS=0 to continue local-only."
    )
elif GCS_ROOT is None:
    print("WARNING: No writable GCS root was found. Outputs will be local only.")
else:
    os.environ["AOU_TGHDL_GCS_ROOT"] = GCS_ROOT

# -----------------------------
# Save/upload helpers.
# -----------------------------
def bq_quote(table_ref):
    """Backtick-quote a BigQuery table reference unless already quoted."""
    table_ref = str(table_ref).strip()
    return table_ref if table_ref.startswith("`") else f"`{table_ref}`"


def _local_to_gcs_path(local_path):
    """Map a local project path to the parallel GCS path under GCS_ROOT."""
    local_path = Path(local_path).resolve()
    try:
        rel = local_path.relative_to(PROJECT_ROOT.resolve())
    except Exception:
        rel = Path(local_path.name)
    if GCS_ROOT is None:
        return None
    return GCS_ROOT.rstrip("/") + "/" + str(rel).replace(os.sep, "/")


def upload_to_bucket(local_path, gcs_path=None, remove_existing=False):
    """Upload a file or directory to GCS. Returns True/False and prints a compact status."""
    local_path = Path(local_path)
    if GCS_ROOT is None and gcs_path is None:
        print(f"No GCS_ROOT; skipping upload for {local_path}")
        return False
    if not local_path.exists():
        print(f"Local path does not exist; skipping upload: {local_path}")
        return False
    if gcs_path is None:
        gcs_path = _local_to_gcs_path(local_path)
    if remove_existing and gcs_path:
        if shutil.which("gcloud"):
            _run(f"gcloud storage rm --recursive '{gcs_path}'", timeout=300)
        elif shutil.which("gsutil"):
            _run(f"gsutil -m rm -r '{gcs_path}'", timeout=300)
    recursive = local_path.is_dir()
    ok, err = _copy_to_gcs(local_path, gcs_path, recursive=recursive, timeout=1800)
    if ok:
        print(f"Uploaded: {local_path} -> {gcs_path}")
        return True
    print(f"Upload failed for {local_path} -> {gcs_path}\n{err}")
    return False


def sync_project_to_gcs(paths=None):
    """Backup standard project output folders to GCS. Safe to rerun."""
    if GCS_ROOT is None:
        print("No GCS_ROOT; sync skipped.")
        return False
    if paths is None:
        paths = [RAW_DIR, INTERMEDIATE_DIR, DERIVED_DIR, QC_DIR, RESULTS_DIR, PLOTS_DIR, LOG_DIR]
    ok_all = True
    for p in paths:
        if Path(p).exists():
            ok_all = upload_to_bucket(p, _local_to_gcs_path(p), remove_existing=False) and ok_all
    return ok_all


def save_json(obj, out_path, auto_upload=AUTO_UPLOAD_SAVED_FILES):
    """Save JSON locally and optionally upload the file immediately."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    if auto_upload and GCS_ROOT:
        upload_to_bucket(out_path)
    return out_path


def save_df(df, out_path, index=False, auto_upload=AUTO_UPLOAD_SAVED_FILES):
    """Save a dataframe locally and optionally upload the saved file immediately."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    saved_path = out_path
    if out_path.suffix.lower() == ".parquet":
        try:
            df.to_parquet(out_path, index=index)
        except Exception as e:
            saved_path = out_path.with_suffix(".csv")
            print(f"Parquet save failed for {out_path.name}; writing CSV fallback. Error: {repr(e)}")
            df.to_csv(saved_path, index=index)
    else:
        df.to_csv(out_path, index=index)
    if auto_upload and GCS_ROOT:
        upload_to_bucket(saved_path)
    return saved_path

run_metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": RUN_ID,
    "project": PROJECT,
    "cdr": CDR,
    "cdr_source": cdr_source,
    "workspace_namespace": WORKSPACE_NAMESPACE,
    "workspace_name": WORKSPACE_NAME,
    "workspace_bucket_env": WORKSPACE_BUCKET_ENV,
    "project_root": str(PROJECT_ROOT),
    "run_dir": str(RUN_DIR),
    "gcs_root": GCS_ROOT,
    "gcs_source": gcs_source,
    "require_gcs_backup": REQUIRE_GCS_BACKUP,
    "auto_upload_saved_files": AUTO_UPLOAD_SAVED_FILES,
    "python": sys.version,
}
save_json(run_metadata, LOG_DIR / "run_metadata.json", auto_upload=bool(GCS_ROOT))

print("SETUP OK")
print("PROJECT:", PROJECT)
print("CDR:", CDR, f"(source: {cdr_source})")
print("WORKSPACE_BUCKET env:", WORKSPACE_BUCKET_ENV)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("GCS_ROOT:", GCS_ROOT, f"(source: {gcs_source})")
print("Diagnostics:")
print(" -", LOG_DIR / "cdr_autodiscovery_diagnostics.csv")
print(" -", LOG_DIR / "gcs_autodiscovery_diagnostics.csv")
print("Helpers available: upload_to_bucket(path), sync_project_to_gcs(), save_df(df, path), save_json(obj, path)")


## Cell 2 — Confirm required OMOP tables

This cell verifies that `person`, `measurement`, and `concept` exist in the resolved CDR.

- `measurement` contains lab values.
- `concept` maps LOINC codes and OMOP concept IDs.
- `person` is useful for optional descriptive joins.


In [ ]:
# Cell 2: Confirm required OMOP tables and inspect schemas.
project_id, dataset_id = CDR.split(".", 1)
dataset_ref = bigquery.DatasetReference(project_id, dataset_id)

tables = list(client.list_tables(dataset_ref))
table_df = pd.DataFrame({"table_id": [t.table_id for t in tables]})
print(f"Found {len(table_df)} tables in {CDR}")
display(table_df.sort_values("table_id").head(80))

required = {"person", "measurement", "concept"}
found = set(table_df["table_id"])
missing = sorted(required - found)
if missing:
    raise ValueError(f"Missing required tables in {CDR}: {missing}")

schema_rows = []
for table_name in sorted(required):
    table = client.get_table(f"{CDR}.{table_name}")
    for field in table.schema:
        schema_rows.append({"table": table_name, "column": field.name, "type": field.field_type, "mode": field.mode})
schema_df = pd.DataFrame(schema_rows)
display(schema_df)

save_df(table_df.sort_values("table_id"), LOG_DIR / "cdr_tables.csv")
save_df(schema_df, LOG_DIR / "required_table_schemas.csv")


## Cell 3 — Define TG and HDL target LOINC codes

This tutorial starts conservatively with standard LOINC codes for serum/plasma triglycerides and HDL cholesterol.

Why query LOINC codes instead of hard-coding OMOP `concept_id`s? Because `concept_id`s can be checked directly in the active CDR, and the concept lookup provides human-readable names and validity metadata.

You can add additional CDR-specific codes later only after inspecting the fuzzy-discovery table.


In [ ]:
# Cell 3: Define target lipid LOINC codes.
TARGET_LOINC = {
    "tg": [
        "2571-8",   # Triglyceride [Mass/volume] in Serum or Plasma
        "14927-8",  # Triglyceride [Moles/volume] in Serum or Plasma
    ],
    "hdl": [
        "2085-9",   # Cholesterol in HDL [Mass/volume] in Serum or Plasma
        "14646-4",  # Cholesterol in HDL [Moles/volume] in Serum or Plasma
        "35197-3",  # Cholesterol in HDL [Mass or Moles/volume] in Serum or Plasma
    ],
}

loinc_df = pd.DataFrame(
    [{"analyte": analyte, "concept_code": code} for analyte, codes in TARGET_LOINC.items() for code in codes]
)
display(loinc_df)


## Cell 4 — Find TG/HDL concepts in the active CDR

This cell looks up the target LOINC codes in the CDR `concept` table and builds `TG_CONCEPT_IDS` and `HDL_CONCEPT_IDS` for extraction.


In [ ]:
# Cell 4: Query the concept table for target LOINC concepts.
loinc_code_list = sorted(set(loinc_df["concept_code"]))
loinc_code_sql = ", ".join([f"'{x}'" for x in loinc_code_list])

concept_sql = f"""
SELECT
  concept_id,
  concept_name,
  domain_id,
  vocabulary_id,
  concept_class_id,
  standard_concept,
  concept_code,
  valid_start_date,
  valid_end_date,
  invalid_reason
FROM {bq_quote(CDR + '.concept')}
WHERE vocabulary_id = 'LOINC'
  AND concept_code IN ({loinc_code_sql})
ORDER BY concept_code
"""
concept_df = client.query(concept_sql).result().to_dataframe()
concept_df = concept_df.merge(loinc_df, on="concept_code", how="left")
display(concept_df)

save_df(concept_df, QC_DIR / "tg_hdl_target_concepts.csv")

TG_CONCEPT_IDS = concept_df.loc[concept_df["analyte"] == "tg", "concept_id"].dropna().astype(int).tolist()
HDL_CONCEPT_IDS = concept_df.loc[concept_df["analyte"] == "hdl", "concept_id"].dropna().astype(int).tolist()
ALL_LIPID_CONCEPT_IDS = sorted(set(TG_CONCEPT_IDS + HDL_CONCEPT_IDS))

print("TG_CONCEPT_IDS:", TG_CONCEPT_IDS)
print("HDL_CONCEPT_IDS:", HDL_CONCEPT_IDS)
print("ALL_LIPID_CONCEPT_IDS:", ALL_LIPID_CONCEPT_IDS)

if not TG_CONCEPT_IDS:
    raise ValueError("No TG concept IDs found. Inspect concept_df and consider adding appropriate LOINC codes.")
if not HDL_CONCEPT_IDS:
    raise ValueError("No HDL concept IDs found. Inspect concept_df and consider adding appropriate LOINC codes.")


## Optional Cell 4b — Fuzzy lipid concept discovery

Run this when you want to inspect nearby TG/HDL concepts in the active CDR. Do **not** automatically add fuzzy matches to the phenotype. First check the concept name, unit distribution, and row counts.


In [ ]:
# Optional Cell 4b: Fuzzy discovery of nearby lipid concepts.
fuzzy_sql = f"""
SELECT
  concept_id,
  concept_name,
  domain_id,
  vocabulary_id,
  concept_class_id,
  standard_concept,
  concept_code,
  valid_start_date,
  valid_end_date,
  invalid_reason
FROM {bq_quote(CDR + '.concept')}
WHERE vocabulary_id = 'LOINC'
  AND domain_id = 'Measurement'
  AND standard_concept = 'S'
  AND invalid_reason IS NULL
  AND (
    LOWER(concept_name) LIKE '%triglyceride%serum%plasma%'
    OR LOWER(concept_name) LIKE '%cholesterol in hdl%serum%plasma%'
    OR LOWER(concept_name) LIKE '%hdl cholesterol%serum%plasma%'
  )
ORDER BY concept_name
LIMIT 500
"""
fuzzy_concepts = client.query(fuzzy_sql).result().to_dataframe()
display(fuzzy_concepts)
save_df(fuzzy_concepts, QC_DIR / "tg_hdl_fuzzy_lipid_concepts.csv")


## Cell 5 — Count TG/HDL measurement rows by concept and unit

Before extraction, inspect unit distributions. TG:HDL is only meaningful after TG and HDL are put on compatible units.

This notebook converts:

- TG mmol/L → mg/dL using `× 88.57`
- HDL mmol/L → mg/dL using `× 38.67`
- mg/dL values remain unchanged

Rows with missing or unrecognized units are excluded in the main extraction.


In [ ]:
# Cell 5: Measurement counts by analyte, concept, and unit.
concept_id_sql = ", ".join(map(str, ALL_LIPID_CONCEPT_IDS))

tg_ids_sql = ", ".join(map(str, TG_CONCEPT_IDS))
hdl_ids_sql = ", ".join(map(str, HDL_CONCEPT_IDS))

counts_sql = f"""
WITH lipid_concepts AS (
  SELECT concept_id, 'tg' AS analyte FROM UNNEST([{tg_ids_sql}]) AS concept_id
  UNION ALL
  SELECT concept_id, 'hdl' AS analyte FROM UNNEST([{hdl_ids_sql}]) AS concept_id
)
SELECT
  lc.analyte,
  m.measurement_concept_id,
  meas.concept_name AS measurement_name,
  m.unit_concept_id,
  unit.concept_name AS unit_name,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT m.person_id) AS n_persons,
  COUNTIF(m.value_as_number IS NOT NULL) AS n_numeric_rows,
  MIN(m.value_as_number) AS min_value,
  APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(50)] AS median_value,
  MAX(m.value_as_number) AS max_value
FROM {bq_quote(CDR + '.measurement')} m
JOIN lipid_concepts lc
  ON m.measurement_concept_id = lc.concept_id
LEFT JOIN {bq_quote(CDR + '.concept')} meas
  ON m.measurement_concept_id = meas.concept_id
LEFT JOIN {bq_quote(CDR + '.concept')} unit
  ON m.unit_concept_id = unit.concept_id
GROUP BY 1,2,3,4,5
ORDER BY analyte, n_rows DESC
"""
unit_counts = client.query(counts_sql).result().to_dataframe()
display(unit_counts)
save_df(unit_counts, QC_DIR / "tg_hdl_unit_counts.csv")


## Cell 6 — Extract same-day TG and HDL pairs

This is the main extraction cell.

Design choices:

- Use `measurement.value_as_number` only.
- Convert recognizable mg/dL and mmol/L units to mg/dL.
- Apply broad biological plausibility filters before pairing.
- Aggregate repeated same-day values using `APPROX_QUANTILES(...)[OFFSET(50)]` in BigQuery.
- Pair TG and HDL only when both are available for the same `person_id` and `lab_date`.

The thresholds are intentionally broad. Tighten them later only if your analysis plan requires it.


In [ ]:
# Cell 6: Extract same-day TG and HDL pairs.

# Broad plausibility thresholds in mg/dL.
TG_MIN_MGDL = 20
TG_MAX_MGDL = 2000
HDL_MIN_MGDL = 5
HDL_MAX_MGDL = 200
RATIO_MIN = 0.05
RATIO_MAX = 50

extract_sql = f"""
WITH lipid_concepts AS (
  SELECT concept_id, 'tg' AS analyte FROM UNNEST([{tg_ids_sql}]) AS concept_id
  UNION ALL
  SELECT concept_id, 'hdl' AS analyte FROM UNNEST([{hdl_ids_sql}]) AS concept_id
),
raw_lipids AS (
  SELECT
    m.person_id,
    COALESCE(m.measurement_date, DATE(m.measurement_datetime)) AS lab_date,
    lc.analyte,
    m.measurement_concept_id,
    meas.concept_name AS measurement_name,
    m.value_as_number,
    m.unit_concept_id,
    unit.concept_name AS unit_name,
    CASE
      WHEN lc.analyte = 'tg'
        AND REGEXP_CONTAINS(LOWER(COALESCE(unit.concept_name, '')), r'mg/dl|milligram per deciliter')
        THEN m.value_as_number
      WHEN lc.analyte = 'tg'
        AND REGEXP_CONTAINS(LOWER(COALESCE(unit.concept_name, '')), r'mmol/l|millimole per liter')
        THEN m.value_as_number * 88.57
      WHEN lc.analyte = 'hdl'
        AND REGEXP_CONTAINS(LOWER(COALESCE(unit.concept_name, '')), r'mg/dl|milligram per deciliter')
        THEN m.value_as_number
      WHEN lc.analyte = 'hdl'
        AND REGEXP_CONTAINS(LOWER(COALESCE(unit.concept_name, '')), r'mmol/l|millimole per liter')
        THEN m.value_as_number * 38.67
      ELSE NULL
    END AS value_mgdl
  FROM {bq_quote(CDR + '.measurement')} m
  JOIN lipid_concepts lc
    ON m.measurement_concept_id = lc.concept_id
  LEFT JOIN {bq_quote(CDR + '.concept')} meas
    ON m.measurement_concept_id = meas.concept_id
  LEFT JOIN {bq_quote(CDR + '.concept')} unit
    ON m.unit_concept_id = unit.concept_id
  WHERE m.value_as_number IS NOT NULL
    AND COALESCE(m.measurement_date, DATE(m.measurement_datetime)) IS NOT NULL
),
clean_lipids AS (
  SELECT *
  FROM raw_lipids
  WHERE value_mgdl IS NOT NULL
    AND (
      (analyte = 'tg' AND value_mgdl BETWEEN {TG_MIN_MGDL} AND {TG_MAX_MGDL})
      OR
      (analyte = 'hdl' AND value_mgdl BETWEEN {HDL_MIN_MGDL} AND {HDL_MAX_MGDL})
    )
),
daily_lipids AS (
  SELECT
    person_id,
    lab_date,
    analyte,
    APPROX_QUANTILES(value_mgdl, 100)[OFFSET(50)] AS value_mgdl,
    COUNT(*) AS n_measurements_same_day,
    STRING_AGG(DISTINCT CAST(measurement_concept_id AS STRING), ',') AS measurement_concept_ids,
    STRING_AGG(DISTINCT measurement_name, ' | ') AS measurement_names,
    STRING_AGG(DISTINCT COALESCE(unit_name, 'missing'), ' | ') AS unit_names
  FROM clean_lipids
  GROUP BY person_id, lab_date, analyte
),
paired AS (
  SELECT
    tg.person_id,
    tg.lab_date,
    tg.value_mgdl AS tg_mgdl,
    hdl.value_mgdl AS hdl_mgdl,
    SAFE_DIVIDE(tg.value_mgdl, hdl.value_mgdl) AS tg_hdl_ratio,
    LOG(SAFE_DIVIDE(tg.value_mgdl, hdl.value_mgdl)) AS log_tg_hdl_ratio,
    tg.n_measurements_same_day AS tg_n_measurements_same_day,
    hdl.n_measurements_same_day AS hdl_n_measurements_same_day,
    tg.measurement_concept_ids AS tg_measurement_concept_ids,
    hdl.measurement_concept_ids AS hdl_measurement_concept_ids,
    tg.measurement_names AS tg_measurement_names,
    hdl.measurement_names AS hdl_measurement_names,
    tg.unit_names AS tg_unit_names,
    hdl.unit_names AS hdl_unit_names
  FROM daily_lipids tg
  JOIN daily_lipids hdl
    ON tg.person_id = hdl.person_id
   AND tg.lab_date = hdl.lab_date
  WHERE tg.analyte = 'tg'
    AND hdl.analyte = 'hdl'
)
SELECT *
FROM paired
WHERE tg_hdl_ratio BETWEEN {RATIO_MIN} AND {RATIO_MAX}
ORDER BY person_id, lab_date
"""

paired_labs = client.query(extract_sql).result().to_dataframe()
print("Same-day paired TG/HDL rows:", len(paired_labs))
print("Unique participants:", paired_labs["person_id"].nunique() if not paired_labs.empty else 0)
display(paired_labs.head())

save_df(paired_labs, DERIVED_DIR / "tg_hdl_same_day_pairs.csv")
save_df(paired_labs, DERIVED_DIR / "tg_hdl_same_day_pairs.parquet")


## Cell 7 — Quick extraction QC

Check the date range, number of repeated measurements, and basic phenotype distributions before making person-level files.


In [ ]:
# Cell 7: Basic extraction QC.
if paired_labs.empty:
    raise ValueError("No paired TG/HDL lab rows were extracted. Check concepts, units, and thresholds.")

paired_labs["lab_date"] = pd.to_datetime(paired_labs["lab_date"])

qc_summary = pd.DataFrame([{
    "n_pair_rows": len(paired_labs), 
    "n_persons": paired_labs["person_id"].nunique(),
    "min_lab_date": paired_labs["lab_date"].min(),
    "max_lab_date": paired_labs["lab_date"].max(),
    "median_tg_mgdl": paired_labs["tg_mgdl"].median(),
    "median_hdl_mgdl": paired_labs["hdl_mgdl"].median(),
    "median_tg_hdl_ratio": paired_labs["tg_hdl_ratio"].median(),
    "median_log_tg_hdl_ratio": paired_labs["log_tg_hdl_ratio"].median(),
}])

display(qc_summary)
save_df(qc_summary, QC_DIR / "tg_hdl_extraction_summary.csv")

pairs_per_person = paired_labs.groupby("person_id").size().rename("n_tg_hdl_pair_dates").reset_index()
display(pairs_per_person["n_tg_hdl_pair_dates"].describe().to_frame())
save_df(pairs_per_person, QC_DIR / "tg_hdl_pairs_per_person.csv")


## Cell 8 — Build person-level TG:HDL phenotypes

For downstream GWAS/ExWAS, you usually want one row per person.

This cell creates two phenotype definitions:

1. **Latest paired phenotype:** most recent same-day TG/HDL pair per person.
2. **Median paired phenotype:** median TG, HDL, and ratio across all valid same-day pairs per person.

For genetic analyses, the median phenotype is often more stable when repeated labs exist. For baseline or time-to-event designs, earliest/latest may be more appropriate depending on the index date.


In [ ]:
# Cell 8: Build person-level latest and median phenotypes.
paired_labs = paired_labs.sort_values(["person_id", "lab_date"]).copy()

# Latest same-day pair per person.
phenotype_latest = paired_labs.groupby("person_id", as_index=False).tail(1).copy()
phenotype_latest = phenotype_latest.rename(columns={
    "lab_date": "tg_hdl_latest_lab_date",
    "tg_mgdl": "tg_mgdl_latest",
    "hdl_mgdl": "hdl_mgdl_latest",
    "tg_hdl_ratio": "tg_hdl_ratio_latest",
    "log_tg_hdl_ratio": "log_tg_hdl_ratio_latest",
})
phenotype_latest["phenotype_source"] = "latest_same_day_pair"

# Median across all same-day pairs per person.
phenotype_median = paired_labs.groupby("person_id", as_index=False).agg(
    n_tg_hdl_pair_dates=("lab_date", "nunique"),
    first_tg_hdl_lab_date=("lab_date", "min"),
    last_tg_hdl_lab_date=("lab_date", "max"),
    tg_mgdl_median=("tg_mgdl", "median"),
    hdl_mgdl_median=("hdl_mgdl", "median"),
    tg_hdl_ratio_median=("tg_hdl_ratio", "median"),
    log_tg_hdl_ratio_median=("log_tg_hdl_ratio", "median"),
)
phenotype_median["phenotype_source"] = "median_same_day_pairs"

# Also compute log of median TG:HDL. This can differ slightly from median of log ratios.
phenotype_median["log_ratio_of_median_tg_hdl"] = np.log(
    phenotype_median["tg_mgdl_median"] / phenotype_median["hdl_mgdl_median"]
)

display(phenotype_median.head())
display(phenotype_latest.head())

save_df(phenotype_median, DERIVED_DIR / "tg_hdl_person_median.csv")
save_df(phenotype_latest, DERIVED_DIR / "tg_hdl_person_latest.csv")


## Cell 9 — inverse-rank normal transformation

For GWAS with a linear model, you may want an inverse-rank normal transformed phenotype. This cell applies a Blom-style rank transformation to the log TG:HDL phenotype.

Use the raw/log-interpretable phenotype for descriptive tables and effect-size interpretation; use IRNT only when it matches your GWAS analysis plan.


In [ ]:
# Cell 9: IRNT phenotype columns.
try:
    from scipy.stats import norm
except Exception as e:
    raise ImportError("scipy is required for IRNT. Install scipy or skip this cell.") from e


def irnt(series):
    x = pd.Series(series).astype(float)
    mask = x.notna() & np.isfinite(x)
    out = pd.Series(np.nan, index=x.index, dtype=float)
    ranks = x.loc[mask].rank(method="average")
    n = int(mask.sum())
    out.loc[mask] = norm.ppf((ranks - 0.5) / n)
    return out

phenotype_median["log_tg_hdl_ratio_median_irnt"] = irnt(phenotype_median["log_tg_hdl_ratio_median"])
phenotype_median["log_ratio_of_median_tg_hdl_irnt"] = irnt(phenotype_median["log_ratio_of_median_tg_hdl"])
phenotype_latest["log_tg_hdl_ratio_latest_irnt"] = irnt(phenotype_latest["log_tg_hdl_ratio_latest"])

save_df(phenotype_median, DERIVED_DIR / "tg_hdl_person_median_with_irnt.csv")
save_df(phenotype_latest, DERIVED_DIR / "tg_hdl_person_latest_with_irnt.csv")

display(phenotype_median[["person_id", "tg_hdl_ratio_median", "log_tg_hdl_ratio_median", "log_tg_hdl_ratio_median_irnt"]].head())


## Cell 10 — descriptive demographics join

This is useful for QC summaries. For GWAS covariates, genetic ancestry and genetic PCs should come from genomics/QC files, not the `person` table.


In [ ]:
# Cell 10: Optional join to person demographics for descriptive QC.
person_sql = f"""
SELECT
  p.person_id,
  p.birth_datetime,
  p.sex_at_birth_concept_id,
  sex.concept_name AS sex_at_birth,
  p.gender_concept_id,
  gender.concept_name AS gender,
  p.race_concept_id,
  race.concept_name AS race,
  p.ethnicity_concept_id,
  ethnicity.concept_name AS ethnicity
FROM {bq_quote(CDR + '.person')} p
LEFT JOIN {bq_quote(CDR + '.concept')} sex
  ON p.sex_at_birth_concept_id = sex.concept_id
LEFT JOIN {bq_quote(CDR + '.concept')} gender
  ON p.gender_concept_id = gender.concept_id
LEFT JOIN {bq_quote(CDR + '.concept')} race
  ON p.race_concept_id = race.concept_id
LEFT JOIN {bq_quote(CDR + '.concept')} ethnicity
  ON p.ethnicity_concept_id = ethnicity.concept_id
"""
person_demo = client.query(person_sql).result().to_dataframe()

phenotype_median_demo = phenotype_median.merge(person_demo, on="person_id", how="left")
phenotype_latest_demo = phenotype_latest.merge(person_demo, on="person_id", how="left")

save_df(phenotype_median_demo, DERIVED_DIR / "tg_hdl_person_median_with_irnt_demographics.csv")
save_df(phenotype_latest_demo, DERIVED_DIR / "tg_hdl_person_latest_with_irnt_demographics.csv")

display(phenotype_median_demo.head())
print("Median phenotype rows:", len(phenotype_median_demo))
print("Latest phenotype rows:", len(phenotype_latest_demo))


## Cell 11 — Quick QC plots

These quick histograms help verify that the phenotype looks reasonable before using it downstream.


In [ ]:
# Cell 11: Quick QC plots.
import matplotlib.pyplot as plt

plot_df = phenotype_median.copy()

plt.figure(figsize=(7, 4))
plt.hist(plot_df["tg_hdl_ratio_median"].dropna(), bins=80)
plt.xlabel("Median TG:HDL ratio")
plt.ylabel("Number of participants")
plt.title("TG:HDL ratio distribution")
plt.tight_layout()
plot_path = PLOTS_DIR / "tg_hdl_ratio_median_hist.png"
plt.savefig(plot_path, dpi=200)
plt.show()
upload_to_bucket(plot_path)

plt.figure(figsize=(7, 4))
plt.hist(plot_df["log_tg_hdl_ratio_median"].dropna(), bins=80)
plt.xlabel("Median log(TG:HDL ratio)")
plt.ylabel("Number of participants")
plt.title("log TG:HDL ratio distribution")
plt.tight_layout()
plot_path = PLOTS_DIR / "log_tg_hdl_ratio_median_hist.png"
plt.savefig(plot_path, dpi=200)
plt.show()
upload_to_bucket(plot_path)

if "log_tg_hdl_ratio_median_irnt" in plot_df.columns:
    plt.figure(figsize=(7, 4))
    plt.hist(plot_df["log_tg_hdl_ratio_median_irnt"].dropna(), bins=80)
    plt.xlabel("IRNT log TG:HDL")
    plt.ylabel("Number of participants")
    plt.title("IRNT phenotype distribution")
    plt.tight_layout()
    plot_path = PLOTS_DIR / "log_tg_hdl_ratio_median_irnt_hist.png"
    plt.savefig(plot_path, dpi=200)
    plt.show()
    upload_to_bucket(plot_path)


## Cell 12 — Final sync and output checklist

Run this at the end to copy standard folders to the discovered GCS root.

Main outputs:

- `data/derived/tg_hdl_same_day_pairs.csv`
- `data/derived/tg_hdl_person_median.csv`
- `data/derived/tg_hdl_person_latest.csv`
- `data/derived/tg_hdl_person_median_with_irnt.csv`
- `data/derived/tg_hdl_person_latest_with_irnt.csv`
- `qc/tg_hdl_unit_counts.csv`
- `qc/tg_hdl_extraction_summary.csv`
- `logs/run_metadata.json`

Recommended downstream phenotype columns:

- Interpretable: `log_tg_hdl_ratio_median` or `tg_hdl_ratio_median`
- GWAS-normalized: `log_tg_hdl_ratio_median_irnt`
- Repeated-lab QC: `n_tg_hdl_pair_dates`, `first_tg_hdl_lab_date`, `last_tg_hdl_lab_date`


In [ ]:
# Cell 12: Final sync and output checklist.
final_files = [
    DERIVED_DIR / "tg_hdl_same_day_pairs.csv",
    DERIVED_DIR / "tg_hdl_person_median.csv",
    DERIVED_DIR / "tg_hdl_person_latest.csv",
    DERIVED_DIR / "tg_hdl_person_median_with_irnt.csv",
    DERIVED_DIR / "tg_hdl_person_latest_with_irnt.csv",
    QC_DIR / "tg_hdl_unit_counts.csv",
    QC_DIR / "tg_hdl_extraction_summary.csv",
    LOG_DIR / "run_metadata.json",
]

print("Local output checklist:")
for p in final_files:
    print("  ", "OK" if p.exists() else "MISSING", p)

print("\nGCS_ROOT:", GCS_ROOT)
if GCS_ROOT:
    sync_project_to_gcs()
    print("\nSynced project outputs to:", GCS_ROOT)
else:
    print("\nNo GCS_ROOT found. Outputs are local only. Set AOU_TGHDL_GCS_ROOT and rerun Cell 1 if needed.")
